# Taxi Fare Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `fare_amount`

## 2 · Project Overview

We predict **taxi fare amounts** based on trip distance, duration, pickup time, borough, passenger count, and surcharge conditions (rush hour, overnight).

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given a taxi trip's distance, duration, pickup time and location, passenger count, and time-based surcharges, predict the metered fare amount.

## 5 · Why This Project Matters

- **Fare prediction** enables upfront pricing and rider expectations.
- Understanding fare drivers helps optimize routes and pricing.
- NYC taxi data is a classic ML regression benchmark.
- Demonstrates regression with metering-based linear structure.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 9,000 |
| **Features** | trip_distance_miles, trip_duration_min, pickup_hour, pickup_day, is_weekend, passenger_count, borough, payment_type, is_rush_hour, is_overnight |
| **Target** | fare_amount (continuous, USD) |
| **Range** | ~$3 – $200 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "fare_amount"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

Synthetic dataset: 9,000 taxi trip records with route and timing features.

In [ ]:
np.random.seed(SEED)
n = 9000
trip_distance_miles = np.round(np.random.lognormal(1.0, 0.8, n).clip(0.3, 40), 2)
pickup_hour = np.random.randint(0, 24, n)
pickup_day = np.random.randint(0, 7, n)
is_weekend = (pickup_day >= 5).astype(int)
passenger_count = np.random.choice([1, 2, 3, 4, 5, 6], n, p=[0.55, 0.2, 0.1, 0.08, 0.04, 0.03])
borough = np.random.choice(["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"], n,
                            p=[0.45, 0.2, 0.15, 0.1, 0.1])
borough_surcharge = {"Manhattan": 2.50, "Brooklyn": 0.50, "Queens": 0.0, "Bronx": 0.0, "Staten Island": 1.00}
boro_val = np.array([borough_surcharge[b] for b in borough])

payment_type = np.random.choice(["Credit", "Cash", "App"], n, p=[0.5, 0.3, 0.2])
is_rush_hour = ((pickup_hour >= 7) & (pickup_hour <= 9) | (pickup_hour >= 16) & (pickup_hour <= 19)).astype(int)
is_overnight = ((pickup_hour >= 20) | (pickup_hour <= 5)).astype(int)

# Duration in minutes (based on distance + traffic)
trip_duration_min = np.round(trip_distance_miles * 3.5 * (1 + 0.5 * is_rush_hour) + np.random.normal(3, 2, n), 1).clip(2, 120)

# Fare model (NYC-style metering)
base_fare = 3.00
per_mile = 2.50
per_minute = 0.50
rush_surcharge = 1.50 * is_rush_hour
overnight_surcharge = 0.75 * is_overnight

fare_amount = (base_fare + per_mile * trip_distance_miles + per_minute * trip_duration_min
               + rush_surcharge + overnight_surcharge + boro_val)
fare_amount = np.round(fare_amount + np.random.normal(0, 1.5, n), 2).clip(3.0, 200.0)

df = pd.DataFrame({
    "trip_distance_miles": trip_distance_miles, "trip_duration_min": trip_duration_min,
    "pickup_hour": pickup_hour, "pickup_day": pickup_day, "is_weekend": is_weekend,
    "passenger_count": passenger_count, "borough": borough,
    "payment_type": payment_type, "is_rush_hour": is_rush_hour,
    "is_overnight": is_overnight, "fare_amount": fare_amount
})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['fare_amount'].describe()}")
df.head()

## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["trip_distance_miles"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Trip Distance (mi)"); axes[0][0].set_ylabel("Fare ($)")
axes[0][0].set_title("Distance vs Fare")

axes[0][1].scatter(df["trip_duration_min"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("Duration (min)"); axes[0][1].set_ylabel("Fare ($)")
axes[0][1].set_title("Duration vs Fare")

df.groupby("pickup_hour")[TARGET].mean().plot(ax=axes[0][2], marker="o", color="steelblue")
axes[0][2].set_title("Avg Fare by Pickup Hour")
axes[0][2].set_xlabel("Hour"); axes[0][2].set_ylabel("Avg Fare ($)")

df.boxplot(column=TARGET, by="borough", ax=axes[1][0])
axes[1][0].set_title("Fare by Borough")
axes[1][0].tick_params(axis="x", rotation=45)

df.boxplot(column=TARGET, by="is_rush_hour", ax=axes[1][1])
axes[1][1].set_title("Fare: Rush Hour vs Normal")

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1][2], square=True, cbar_kws={"shrink": 0.8})
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `fare_amount`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Fare Amount ($)")

axes[1].boxplot(df[TARGET], vert=True)
axes[1].set_title(f"Box Plot of {TARGET}")
axes[1].set_ylabel("Fare ($)")

plt.tight_layout()
plt.show()
print(f"Range: ${df[TARGET].min():.2f} to ${df[TARGET].max():.2f}")
print(f"Mean: ${df[TARGET].mean():.2f}, Median: ${df[TARGET].median():.2f}")
print(f"Skew: {df[TARGET].skew():.2f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `borough`, `payment_type` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: Long-distance trips create right skew — keep them.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["speed_mph"] = X_train["trip_distance_miles"] / (X_train["trip_duration_min"] / 60 + 0.01)
X_test["speed_mph"] = X_test["trip_distance_miles"] / (X_test["trip_duration_min"] / 60 + 0.01)

X_train["hour_sin"] = np.sin(2 * np.pi * X_train["pickup_hour"] / 24)
X_test["hour_sin"] = np.sin(2 * np.pi * X_test["pickup_hour"] / 24)

X_train["hour_cos"] = np.cos(2 * np.pi * X_train["pickup_hour"] / 24)
X_test["hour_cos"] = np.cos(2 * np.pi * X_test["pickup_hour"] / 24)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Trip distance** is the strongest fare predictor (per-mile charge dominates).
- **Trip duration** adds per-minute metering component.
- **Rush hour** adds a $1.50 surcharge.
- **Borough surcharges** (Manhattan: $2.50) are clearly captured.
- **Overnight trips** have a small surcharge.

**Business takeaway:** Fares are mostly distance-driven with time and surcharge overlays. Linear models work well because taxi meters are inherently linear.

## 26 · Limitations

1. Synthetic data based on NYC taxi fare structure.
2. No GPS coordinates or actual routing.
3. Missing toll charges.
4. No surge/dynamic pricing.
5. Simplified borough surcharges.

## 27 · How to Improve This Project

1. Use real NYC TLC trip data.
2. Add GPS pickup/dropoff coordinates.
3. Include tolls and airport surcharges.
4. Model ride-share dynamic pricing.
5. Add weather and event features.

## 28 · Production Considerations

- Deploy for upfront fare estimation.
- Monitor for fare anomalies (driver fraud).
- Compare model estimates against meter.
- Update with tariff changes.
- Output fare ranges for rider display.

## 29 · Common Mistakes

1. Not accounting for the dual metering (distance + time).
2. Ignoring borough-specific surcharges.
3. Using only distance without duration (traffic matters).
4. Not handling short-trip minimum fares.
5. Treating rush hour as a continuous rather than binary feature.

## 30 · Mini Challenge / Exercises

1. Use only `trip_distance_miles` — how close to the full model?
2. Remove rush-hour features — how much does RMSE increase?
3. Calculate the effective per-mile rate by borough.
4. Plot fare vs. distance, colored by rush hour.
5. Build separate models for Manhattan vs. outer boroughs.

## 31 · Final Summary / Key Takeaways

1. **Trip distance** is the dominant fare driver.
2. **Trip duration** adds per-minute metering.
3. **Rush hour and overnight** surcharges are clearly captured.
4. **Borough surcharges** add location-specific premiums.
5. **Linear models** work well (metering is inherently linear).
6. **Real fare prediction** needs GPS, tolls, and dynamic pricing.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))